[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_12/SFM_ch12_validation_irb/SFM_ch12_validation_irb.ipynb)

# SFM_ch12_validation_irb

Validation of the scorecard on the test sample: confusion matrix at the cut-offs 0.5 and 1/6 (costs 5:1), expected cost against the cut-off, ROC curves and AUC with DeLong standard errors and tests, Gini, CAP curve and accuracy ratio, Kolmogorov-Smirnov statistic, Brier score, Hosmer-Lemeshow test, repeated stratified cross-validation with WoE inside every fold; calibration of a logit on the Taiwan credit card data; expected loss and Basel II IRB capital of retail exposures as functions of PD; the conditional PD of the one-factor Gaussian model in a bad, typical and good state of the economy (as in SFEdefaproba).

*Statistics of Financial Markets (SFM), Chapter 12: Scoring models. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/SFM/Chapter_12'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import sfm_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/SFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the SFM repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years

## Analysis

In [ ]:
CREDIT_RAW = 'https://raw.githubusercontent.com/danpele/SFM/main/data/credit/'

FILES = {'sgc': 'south_german_credit.csv', 'taiwan': 'taiwan_credit_card_default.csv'}

SEED = 2026

TEST_SHARE = 0.3

POP_BAD = 0.05

COST_FN = 5.0

COST_FP = 1.0

PDO, BASE_SCORE, BASE_ODDS = 20.0, 600.0, 50.0

IV_MIN = 0.1

N_REPEAT, N_FOLD = 20, 5

BINS = {'duration': [12, 18, 24, 36, np.inf], 'amount': [1300, 2000, 3000, 5000, np.inf], 'age': [25, 30, 35, 45, np.inf]}

NUMERIC = ['duration', 'amount', 'age']

SGC_VARS = ['status', 'duration', 'credit_history', 'purpose', 'amount', 'savings', 'employment_duration', 'installment_rate', 'personal_status_sex', 'other_debtors', 'present_residence', 'property', 'age', 'other_installment_plans', 'housing', 'number_credits', 'job', 'people_liable', 'telephone', 'foreign_worker']

STATUS = {1: 'no checking account', 2: 'balance < 0 DM', 3: '0 to 200 DM', 4: '>= 200 DM or salary account'}

TW_FEATURES = ['log_limit', 'age', 'pay0_m2', 'pay0_m1', 'pay0_1', 'pay0_2', 'pay0_3', 'late_months', 'util', 'log_pay1', 'log_pay2', 'log_pay3']

COLORS = {'good': '#1A3A6E', 'bad': '#CD0000', 'logit': '#CD0000', 'lda': '#1A3A6E', 'single': '#B5853F', 'small': '#2E7D32', 'random': '#8E44AD', 'perfect': '#E67E22'}

def load_credit(name='sgc'):
    """A credit data set from data/credit (local copy) or from the raw GitHub URL; adds 'default' (1 = bad)."""
    fn = FILES[name]
    local = next((os.path.join(d, 'data', 'credit', fn) for d in ('.', '..', '../..', '../../..')
                  if os.path.exists(os.path.join(d, 'data', 'credit', fn))), None)
    df = pd.read_csv(local or CREDIT_RAW + fn)
    if name == 'sgc':
        df['default'] = 1 - df['credit_risk']
    return df


def taiwan_features(df):
    """Model inputs of the Taiwan data: log limit, age, the repayment status of September 2005 as indicators (-2 no
    consumption, -1 paid in full, 1, 2, 3+ months late; 0 = minimum paid is the reference), the number of late months
    April-August, utilisation (bill / limit, clipped to [-1, 3]) and log payments of the last three months."""
    X = pd.DataFrame(index=df.index)
    X['log_limit'] = np.log(df['LIMIT_BAL'])
    X['age'] = df['AGE']
    p0 = df['PAY_0']
    X['pay0_m2'] = (p0 == -2).astype(int)
    X['pay0_m1'] = (p0 == -1).astype(int)
    X['pay0_1'] = (p0 == 1).astype(int)
    X['pay0_2'] = (p0 == 2).astype(int)
    X['pay0_3'] = (p0 >= 3).astype(int)
    X['late_months'] = sum((df[c] >= 1).astype(int) for c in ['PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6'])
    X['util'] = (df['BILL_AMT1'] / df['LIMIT_BAL']).clip(-1, 3)
    for i in (1, 2, 3):
        X[f'log_pay{i}'] = np.log1p(df[f'PAY_AMT{i}'].clip(lower=0))
    return X


def stratified_split(y, share=TEST_SHARE, seed=SEED):
    """Indices of a stratified train/test split (the same share of bads in both samples)."""
    rng = np.random.default_rng(seed)
    y = np.asarray(y)
    test = []
    for c in (0, 1):
        idx = np.flatnonzero(y == c)
        rng.shuffle(idx)
        test.append(idx[:int(round(share * len(idx)))])
    test = np.sort(np.concatenate(test))
    train = np.setdiff1d(np.arange(len(y)), test)
    return train, test


def bin_values(x, var):
    """Bin labels of a variable: fixed bins for duration, amount and age, the category code otherwise."""
    if var in BINS:
        edges = [-np.inf] + BINS[var]
        return pd.cut(x, edges, right=True, labels=False).astype(int)
    return x.astype(int)


def woe_table(x, y, var):
    """WoE_i = ln(share of goods in bin i / share of bads in bin i); IV = sum (g_i/G - b_i/B) WoE_i.
    Bins without goods or without bads get 0.5 added to both counts (no infinite WoE)."""
    b = bin_values(pd.Series(x).reset_index(drop=True), var)
    y = pd.Series(np.asarray(y))
    t = pd.DataFrame({'bin': b, 'y': y}).groupby('bin')['y'].agg(n='size', bad='sum')
    t['good'] = t['n'] - t['bad']
    adj = ((t['good'] == 0) | (t['bad'] == 0)) * 0.5
    g, bd = t['good'] + adj, t['bad'] + adj
    t['bad_rate'] = t['bad'] / t['n']
    t['dist_good'] = g / g.sum()
    t['dist_bad'] = bd / bd.sum()
    t['woe'] = np.log(t['dist_good'] / t['dist_bad'])
    t['iv'] = (t['dist_good'] - t['dist_bad']) * t['woe']
    return t


def iv_all(df, y, variables=SGC_VARS):
    """Information value of every variable, largest first."""
    return pd.Series({v: woe_table(df[v], y, v)['iv'].sum() for v in variables}).sort_values(ascending=False)


def iv_strength(iv):
    """Rule of thumb (Siddiqi, 2006): < 0.02 useless, 0.02-0.1 weak, 0.1-0.3 medium, >= 0.3 strong."""
    return 'useless' if iv < 0.02 else 'weak' if iv < 0.1 else 'medium' if iv < 0.3 else 'strong'


def woe_transform(df, tables, variables):
    """Replace every value by the WoE of its bin (tables estimated on the training sample)."""
    out = pd.DataFrame(index=df.index)
    for v in variables:
        b = bin_values(df[v], v)
        out[v] = b.map(tables[v]['woe']).fillna(0.0).values
    return out


def logit_fit(X, y):
    """Logistic regression by maximum likelihood (Newton-Raphson): ln(p/(1-p)) = b0 + x'b.
    Returns the coefficients (constant first), standard errors and the log-likelihood."""
    X = np.column_stack([np.ones(len(X)), np.asarray(X, float)])
    y = np.asarray(y, float)
    b = np.zeros(X.shape[1])
    for _ in range(100):
        p = 1 / (1 + np.exp(-X @ b))
        W = p * (1 - p)
        H = X.T @ (X * W[:, None])
        step = np.linalg.solve(H, X.T @ (y - p))
        b += step
        if np.max(np.abs(step)) < 1e-10:
            break
    p = 1 / (1 + np.exp(-X @ b))
    H = X.T @ (X * (p * (1 - p))[:, None])
    se = np.sqrt(np.diag(np.linalg.inv(H)))
    ll = float(np.sum(y * np.log(p) + (1 - y) * np.log(1 - p)))
    return {'b': b, 'se': se, 'll': ll, 'n': len(y)}


def logit_predict(fit, X):
    X = np.column_stack([np.ones(len(X)), np.asarray(X, float)])
    return 1 / (1 + np.exp(-X @ fit['b']))


def lda_fit(X, y):
    """Fisher's linear discriminant: w = S_W^{-1}(m_1 - m_0) maximises the between/within variance ratio of w'x;
    with Normal classes and a common covariance, ln odds(bad | x) = c + w'x (the prior enters only c)."""
    X = np.asarray(X, float)
    y = np.asarray(y)
    m0, m1 = X[y == 0].mean(0), X[y == 1].mean(0)
    n0, n1 = (y == 0).sum(), (y == 1).sum()
    S = ((X[y == 0] - m0).T @ (X[y == 0] - m0) + (X[y == 1] - m1).T @ (X[y == 1] - m1)) / (n0 + n1 - 2)
    w = np.linalg.solve(S, m1 - m0)
    pi1 = n1 / (n0 + n1)
    c = -0.5 * (m1 + m0) @ w + np.log(pi1 / (1 - pi1))
    return {'w': w, 'c': float(c), 'm0': m0, 'm1': m1, 'S': S, 'pi1': float(pi1)}


def lda_predict(fit, X):
    """Posterior probability of default from the LDA log-odds."""
    return 1 / (1 + np.exp(-(fit['c'] + np.asarray(X, float) @ fit['w'])))


def auc(score, y):
    """AUC = P(score of a random bad > score of a random good) (ties count 1/2), via the Mann-Whitney ranks."""
    score, y = np.asarray(score, float), np.asarray(y)
    r = stats.rankdata(score)
    n1, n0 = (y == 1).sum(), (y == 0).sum()
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def delong(score, y, score2=None):
    """DeLong et al. (1988): standard error of the AUC; with a second score on the same cases, the test of
    equal AUCs (z statistic and two-sided p-value)."""
    def comps(s):
        s = np.asarray(s, float)
        pos, neg = s[np.asarray(y) == 1], s[np.asarray(y) == 0]
        v10 = np.array([np.mean((p > neg) + 0.5 * (p == neg)) for p in pos])
        v01 = np.array([np.mean((pos > q) + 0.5 * (pos == q)) for q in neg])
        return v10, v01
    a10, a01 = comps(score)
    A = a10.mean()
    if score2 is None:
        var = np.var(a10, ddof=1) / len(a10) + np.var(a01, ddof=1) / len(a01)
        return {'auc': float(A), 'se': float(np.sqrt(var)), 'lo': float(A - 1.96 * np.sqrt(var)),
                'hi': float(A + 1.96 * np.sqrt(var))}
    b10, b01 = comps(score2)
    S10 = np.cov(np.vstack([a10, b10]))
    S01 = np.cov(np.vstack([a01, b01]))
    var = (S10[0, 0] + S10[1, 1] - 2 * S10[0, 1]) / len(a10) + (S01[0, 0] + S01[1, 1] - 2 * S01[0, 1]) / len(a01)
    z = (A - b10.mean()) / np.sqrt(var)
    return {'auc1': float(A), 'auc2': float(b10.mean()), 'z': float(z), 'p': float(2 * stats.norm.sf(abs(z)))}


def roc_points(score, y):
    """ROC curve: (FPR, TPR) when every score value is used as the cut-off (a higher score = riskier)."""
    score, y = np.asarray(score, float), np.asarray(y)
    thr = np.r_[np.inf, np.unique(score)[::-1]]
    tpr = np.array([np.mean(score[y == 1] >= t) for t in thr])
    fpr = np.array([np.mean(score[y == 0] >= t) for t in thr])
    return fpr, tpr, thr


def cap_points(score, y):
    """CAP (cumulative accuracy profile): share of all bads among the riskiest x% of the applicants."""
    score, y = np.asarray(score, float), np.asarray(y)
    order = np.argsort(-score, kind='mergesort')
    x = np.r_[0, np.arange(1, len(y) + 1) / len(y)]
    hit = np.r_[0, np.cumsum(y[order]) / y.sum()]
    return x, hit


def accuracy_ratio(score, y):
    """AR = (area between the model CAP and the diagonal) / (area between the perfect CAP and the diagonal);
    for a continuous score AR = Gini = 2 AUC - 1."""
    x, hit = cap_points(score, y)
    a_model = np.trapezoid(hit, x) - 0.5
    pbad = np.mean(y)
    a_perf = (1 - pbad / 2) - 0.5
    return float(a_model / a_perf)


def ks_stat(score, y):
    """Kolmogorov-Smirnov: the largest distance between the score distributions of the bads and the goods."""
    score, y = np.asarray(score, float), np.asarray(y)
    s = np.sort(np.unique(score))
    F1 = np.searchsorted(np.sort(score[y == 1]), s, side='right') / (y == 1).sum()
    F0 = np.searchsorted(np.sort(score[y == 0]), s, side='right') / (y == 0).sum()
    i = int(np.argmax(np.abs(F0 - F1)))
    return {'ks': float(np.abs(F0 - F1)[i]), 'at': float(s[i]), 's': s, 'F1': F1, 'F0': F0}


def brier(p, y):
    """Brier score: the mean squared difference between the predicted PD and the outcome (0 or 1)."""
    return float(np.mean((np.asarray(p) - np.asarray(y)) ** 2))


def calibration_table(p, y, k=10):
    """Mean predicted PD and observed default rate in k groups of equal size (sorted by PD); Hosmer-Lemeshow."""
    d = pd.DataFrame({'p': np.asarray(p), 'y': np.asarray(y)}).sort_values('p').reset_index(drop=True)
    d['g'] = np.arange(len(d)) * k // len(d)
    t = d.groupby('g').agg(n=('y', 'size'), pd_mean=('p', 'mean'), rate=('y', 'mean'), bad=('y', 'sum'))
    exp = t['n'] * t['pd_mean']
    hl = float((((t['bad'] - exp) ** 2) / (exp * (1 - t['pd_mean']))).sum())
    return t, hl, float(stats.chi2.sf(hl, k - 2))


def confusion(p, y, cut):
    """Reject (predict bad) if PD > cut. TP: bad rejected; FN: bad accepted; FP: good rejected; TN: good accepted."""
    p, y = np.asarray(p), np.asarray(y)
    pred = (p > cut).astype(int)
    TP = int(np.sum((pred == 1) & (y == 1)))
    FN = int(np.sum((pred == 0) & (y == 1)))
    FP = int(np.sum((pred == 1) & (y == 0)))
    TN = int(np.sum((pred == 0) & (y == 0)))
    n = len(y)
    return {'cut': float(cut), 'TP': TP, 'FN': FN, 'FP': FP, 'TN': TN, 'acc': (TP + TN) / n,
            'tpr': TP / (TP + FN), 'tnr': TN / (TN + FP), 'fpr': FP / (TN + FP), 'precision': TP / max(TP + FP, 1),
            'cost': (COST_FN * FN + COST_FP * FP) / n, 'reject': (TP + FP) / n}


def scaling(pdo=PDO, base=BASE_SCORE, odds=BASE_ODDS):
    """Score = offset + factor ln(good:bad odds): factor = PDO / ln 2, offset = base - factor ln(odds)."""
    factor = pdo / np.log(2)
    return {'factor': float(factor), 'offset': float(base - factor * np.log(odds))}


def points_table(fit, tables, variables):
    """Points of every attribute: -(b_j WoE_ij + b_0/k) factor + offset/k (k variables, logit for the bads);
    the score of an applicant is the sum of its points."""
    sc = scaling()
    k = len(variables)
    rows = []
    for j, v in enumerate(variables):
        for bn, r in tables[v].iterrows():
            pts = -(fit['b'][j + 1] * r['woe'] + fit['b'][0] / k) * sc['factor'] + sc['offset'] / k
            rows.append({'variable': v, 'bin': bn, 'n': int(r['n']), 'bad_rate': r['bad_rate'], 'woe': r['woe'],
                         'points': pts})
    return pd.DataFrame(rows)


def score_from_pd(p):
    """Score of a PD: offset + factor ln((1 - p)/p)."""
    sc = scaling()
    return sc['offset'] + sc['factor'] * np.log((1 - np.asarray(p)) / np.asarray(p))


def prior_shift(sample_bad, pop_bad):
    """Correction of the log-odds for oversampled bads: ln odds_pop = ln odds_sample + ln(odds of pop / sample)."""
    return float(np.log(pop_bad / (1 - pop_bad)) - np.log(sample_bad / (1 - sample_bad)))


def sgc_scorecard(df=None):
    """WoE tables and IV on the training sample, variables with IV >= IV_MIN, WoE logit and LDA; test predictions."""
    df = load_credit('sgc') if df is None else df
    y = df['default'].values
    tr, te = stratified_split(y)
    tables = {v: woe_table(df[v].iloc[tr], y[tr], v) for v in SGC_VARS}
    iv = pd.Series({v: tables[v]['iv'].sum() for v in SGC_VARS}).sort_values(ascending=False)
    sel = [v for v in iv.index if iv[v] >= IV_MIN]
    Xtr, Xte = woe_transform(df.iloc[tr], tables, sel), woe_transform(df.iloc[te], tables, sel)
    lg = logit_fit(Xtr, y[tr])
    ld = lda_fit(Xtr, y[tr])
    out = {'tr': tr, 'te': te, 'y': y, 'tables': tables, 'iv': iv, 'sel': sel, 'logit': lg, 'lda': ld,
           'p_tr': logit_predict(lg, Xtr), 'p_te': logit_predict(lg, Xte), 'q_te': lda_predict(ld, Xte),
           'Xtr': Xtr, 'Xte': Xte}
    out['single_te'] = -woe_transform(df.iloc[te], tables, ['status'])['status'].values   # status alone (-WoE)
    return out


def small_logit(df=None):
    """Interpretable logit on the full sample: duration (years), age (decades), amount (1000 DM) and the two riskiest
    checking-account groups; odds ratios exp(b) with 95% confidence intervals exp(b -+ 1.96 se)."""
    df = load_credit('sgc') if df is None else df
    X = pd.DataFrame({'duration_years': df['duration'] / 12, 'age_decades': df['age'] / 10,
                      'amount_1000': df['amount'] / 1000, 'no_account': (df['status'] == 1).astype(int),
                      'negative_balance': (df['status'] == 2).astype(int)})
    f = logit_fit(X, df['default'])
    names = ['const'] + list(X.columns)
    t = pd.DataFrame({'b': f['b'], 'se': f['se']}, index=names)
    t['z'] = t['b'] / t['se']
    t['p'] = 2 * stats.norm.sf(np.abs(t['z']))
    t['or'] = np.exp(t['b'])
    t['or_lo'] = np.exp(t['b'] - 1.96 * t['se'])
    t['or_hi'] = np.exp(t['b'] + 1.96 * t['se'])
    f0 = logit_fit(np.zeros((len(df), 0)), df['default'])
    return {'table': t, 'll': f['ll'], 'll0': f0['ll'], 'lr': 2 * (f['ll'] - f0['ll']), 'n': f['n'], 'fit': f, 'X': X}


def cv_auc(df=None, n_repeat=N_REPEAT, n_fold=N_FOLD, seed=SEED):
    """Repeated stratified k-fold cross-validation; WoE tables and variable selection are redone inside every
    training fold (otherwise the test folds leak into the bins). Train and test AUC of four models."""
    df = load_credit('sgc') if df is None else df
    y = df['default'].values
    rng = np.random.default_rng(seed)
    res = {m: {'train': [], 'test': []} for m in ['WoE logit', 'LDA', 'small logit', 'status only']}
    Xs = small_logit(df)['X'].values
    for r in range(n_repeat):
        folds = np.empty(len(y), int)
        for c in (0, 1):
            idx = np.flatnonzero(y == c)
            rng.shuffle(idx)
            folds[idx] = np.arange(len(idx)) % n_fold
        for k in range(n_fold):
            tr, te = np.flatnonzero(folds != k), np.flatnonzero(folds == k)
            tabs = {v: woe_table(df[v].iloc[tr], y[tr], v) for v in SGC_VARS}
            sel = [v for v in SGC_VARS if tabs[v]['iv'].sum() >= IV_MIN]
            Xtr, Xte = woe_transform(df.iloc[tr], tabs, sel), woe_transform(df.iloc[te], tabs, sel)
            lg, ld, sm = logit_fit(Xtr, y[tr]), lda_fit(Xtr, y[tr]), logit_fit(Xs[tr], y[tr])
            pairs = {'WoE logit': (logit_predict(lg, Xtr), logit_predict(lg, Xte)),
                     'LDA': (lda_predict(ld, Xtr), lda_predict(ld, Xte)),
                     'small logit': (logit_predict(sm, Xs[tr]), logit_predict(sm, Xs[te])),
                     'status only': (-Xtr.get('status', pd.Series(0, index=Xtr.index)).values,
                                     -Xte.get('status', pd.Series(0, index=Xte.index)).values)}
            for m, (a, b) in pairs.items():
                res[m]['train'].append(auc(a, y[tr]))
                res[m]['test'].append(auc(b, y[te]))
    return res


def validation(S):
    """All test-sample measures of the WoE logit scorecard and of the LDA."""
    y = S['y'][S['te']]
    p, q = S['p_te'], S['q_te']
    out = {'n_test': int(len(y)), 'bad_test': int(y.sum()), 'n_train': int(len(S['tr'])),
           'auc_train': auc(S['p_tr'], S['y'][S['tr']]), 'auc': auc(p, y), 'auc_lda': auc(q, y),
           'auc_single': auc(S['single_te'], y), 'delong': delong(p, y), 'delong_lda': delong(p, y, q),
           'delong_single': delong(p, y, S['single_te']), 'gini': 2 * auc(p, y) - 1, 'ar': accuracy_ratio(p, y),
           'ks': ks_stat(p, y)['ks'], 'ks_score': float(score_from_pd(ks_stat(p, y)['at'])),
           'brier': brier(p, y), 'brier_ref': brier(np.full(len(y), np.mean(S['y'][S['tr']])), y),
           'cm50': confusion(p, y, 0.5), 'cm_cost': confusion(p, y, COST_FP / (COST_FP + COST_FN)),
           'corr_logit_lda': float(np.corrcoef(np.log(p / (1 - p)), np.log(q / (1 - q)))[0, 1])}
    t, hl, php = calibration_table(p, y, 10)
    out['hl'], out['hl_p'] = hl, php
    out['all_good'] = float(np.mean(y == 0))
    return out


def fig_roc(S, save=True):
    """ROC curves on the test sample: WoE logit, Fisher LDA, the checking-account status alone."""
    y = S['y'][S['te']]
    fig, ax = plt.subplots(figsize=(6.4, 5.0))
    for lab, s, c, ls in [('WoE logit', S['p_te'], COLORS['logit'], '-'), ('Fisher LDA', S['q_te'], COLORS['lda'], '--'),
                          ('checking account only', S['single_te'], COLORS['single'], '-')]:
        f, t, _ = roc_points(s, y)
        ax.plot(f, t, color=c, lw=2.2, ls=ls, label=f'{lab} (AUC = {auc(s, y):.3f})')
    ax.plot([0, 1], [0, 1], color=COLORS['random'], ls=':', lw=1.5, label='random score (AUC = 0.5)')
    ax.set_xlabel('false positive rate: goods rejected (1 - specificity)')
    ax.set_ylabel('true positive rate: bads rejected')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    st.legend_outside_bottom(ax, ncol=2, y=-0.16)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch12_roc')


def fig_cap(S, save=True):
    """CAP curves on the test sample: model, random and perfect; AR in the legend."""
    y = S['y'][S['te']]
    p = S['p_te']
    x, h = cap_points(p, y)
    pb = y.mean()
    fig, ax = plt.subplots(figsize=(6.4, 5.0))
    ax.plot(x, h, color=COLORS['logit'], lw=2.2, label=f'WoE logit (AR = {accuracy_ratio(p, y):.3f})')
    ax.plot([0, pb, 1], [0, 1, 1], color=COLORS['perfect'], lw=2, ls='--', label='perfect model (AR = 1)')
    ax.plot([0, 1], [0, 1], color=COLORS['random'], ls=':', lw=1.5, label='random score (AR = 0)')
    ax.fill_between(x, x, h, color=COLORS['logit'], alpha=0.12)
    ax.set_xlabel('share of applicants, riskiest first')
    ax.set_ylabel('share of all bads captured')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    st.legend_outside_bottom(ax, ncol=1, y=-0.16)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch12_cap')


def fig_ks(S, save=True):
    """Empirical distribution functions of the score for goods and bads; the KS distance."""
    y = S['y'][S['te']]
    sc = score_from_pd(S['p_te'])
    k = ks_stat(-sc, y)                     # risk order: a low score is risky
    s = np.sort(np.unique(sc))
    F1 = np.searchsorted(np.sort(sc[y == 1]), s, side='right') / (y == 1).sum()
    F0 = np.searchsorted(np.sort(sc[y == 0]), s, side='right') / (y == 0).sum()
    i = int(np.argmax(F1 - F0))
    fig, ax = plt.subplots(figsize=(9.5, 4.0))
    ax.step(s, F1, where='post', color=COLORS['bad'], lw=2, label='bads: share with score <= s')
    ax.step(s, F0, where='post', color=COLORS['good'], lw=2, label='goods: share with score <= s')
    ax.vlines(s[i], F0[i], F1[i], color=st.Forest, lw=3, label=f'KS = {F1[i] - F0[i]:.3f} at {s[i]:.0f} points')
    ax.set_xlabel('score s (points)')
    ax.set_ylabel('cumulative share')
    st.legend_outside_bottom(ax, ncol=3, y=-0.2)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch12_ks')
    return {'ks': float(F1[i] - F0[i]), 'at': float(s[i]), 'ks_check': k['ks']}


def fig_cost_cutoff(S, save=True):
    """Test sample: expected cost per applicant (5 for an accepted bad, 1 for a rejected good) and the share of
    rejected applicants as functions of the PD cut-off."""
    y = S['y'][S['te']]
    p = S['p_te']
    cuts = np.linspace(0.02, 0.9, 89)
    cost = [confusion(p, y, c)['cost'] for c in cuts]
    rej = [confusion(p, y, c)['reject'] for c in cuts]
    fig, ax = plt.subplots(figsize=(9.5, 4.0))
    ax.plot(cuts, cost, color=st.IDAred, lw=2.2, label='expected cost per applicant')
    ax.axhline(COST_FN * y.mean(), color=st.Amber, ls='--', lw=1.5, label='accept everybody')
    ax.axhline(COST_FP * (1 - y.mean()), color=st.Purple, ls='--', lw=1.5, label='reject everybody')
    ax.axvline(COST_FP / (COST_FP + COST_FN), color=st.Forest, ls=':', lw=2, label='Bayes cut-off PD = 1/6')
    ax.set_xlabel('PD cut-off (reject if PD > cut-off)')
    ax.set_ylabel('cost per applicant')
    ax2 = ax.twinx()
    ax2.plot(cuts, 100 * np.array(rej), color=st.MainBlue, lw=1.8, label='rejected applicants (%, right axis)')
    ax2.set_ylabel('rejected (%)')
    ax2.spines['right'].set_visible(True)
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=3, frameon=False)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch12_cost_cutoff')
    i = int(np.argmin(cost))
    return {'best_cut': float(cuts[i]), 'best_cost': float(cost[i]), 'cost_accept_all': float(COST_FN * y.mean()),
            'cost_reject_all': float(COST_FP * (1 - y.mean()))}


def fig_cv_auc(res, save=True):
    """Repeated cross-validation: train and test AUC of four models (one dot per fold)."""
    models = list(res)
    col = [COLORS['logit'], COLORS['lda'], COLORS['small'], COLORS['single']]
    fig, ax = plt.subplots(figsize=(9.5, 4.0))
    rng = np.random.default_rng(SEED)
    for i, (m, c) in enumerate(zip(models, col)):
        for j, (part, mk) in enumerate([('train', 'o'), ('test', 'x')]):
            v = np.array(res[m][part])
            xpos = i + (j - 0.5) * 0.36 + rng.uniform(-0.07, 0.07, len(v))
            ax.scatter(xpos, v, s=12, color=c, marker=mk, alpha=0.6,
                       label=('training folds' if part == 'train' else 'test folds') if i == 0 else None)
            ax.plot([i + (j - 0.5) * 0.36 - 0.14, i + (j - 0.5) * 0.36 + 0.14], [v.mean()] * 2, color='black', lw=2)
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels(models)
    ax.set_ylabel('AUC')
    st.legend_outside_bottom(ax, ncol=2, y=-0.14)
    leg = ax.get_legend()
    for h in leg.legend_handles:
        h.set_color(st.DarkText)
        h.set_alpha(1)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch12_cv_auc')
    return {m: {p: {'mean': float(np.mean(v)), 'sd': float(np.std(v, ddof=1))} for p, v in d.items()}
            for m, d in res.items()}


def taiwan_model(df=None):
    """Logit on the Taiwan data (70/30 stratified split); sex, education and marital status are not inputs."""
    df = load_credit('taiwan') if df is None else df
    X = taiwan_features(df)
    y = df['default'].values
    tr, te = stratified_split(y)
    mu, sd = X.iloc[tr].mean(), X.iloc[tr].std()
    Z = (X - mu) / sd
    f = logit_fit(Z.iloc[tr], y[tr])
    return {'df': df, 'X': X, 'Z': Z, 'y': y, 'tr': tr, 'te': te, 'fit': f, 'p': logit_predict(f, Z)}


def fig_calibration(T, save=True):
    """Calibration of the Taiwan logit on the test sample: mean PD against the observed rate in ten groups."""
    y, p = T['y'][T['te']], T['p'][T['te']]
    t, hl, php = calibration_table(p, y, 10)
    se = np.sqrt(t['rate'] * (1 - t['rate']) / t['n'])
    fig, ax = plt.subplots(figsize=(6.4, 5.0))
    ax.plot([0, 1], [0, 1], color=COLORS['random'], ls=':', lw=1.5, label='perfect calibration')
    ax.errorbar(t['pd_mean'], t['rate'], yerr=1.96 * se, fmt='o-', color=st.IDAred, ecolor=st.MainBlue, capsize=3,
                lw=2, label='ten groups of 900 test clients (95% interval)')
    ax.set_xlabel('mean predicted PD in the group')
    ax.set_ylabel('observed default rate in the group')
    ax.set_xlim(0, 0.8)
    ax.set_ylim(0, 0.8)
    st.legend_outside_bottom(ax, ncol=1, y=-0.16)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch12_calibration')
    return {'hl': hl, 'hl_p': php, 'brier': brier(p, y), 'brier_ref': brier(np.full(len(y), T['y'][T['tr']].mean()), y),
            'auc': auc(p, y), 'auc_train': auc(T['p'][T['tr']], T['y'][T['tr']]), 'n_test': int(len(y)),
            'rate': float(y.mean()), 'groups': t.round(4).to_dict('index')}


def irb_capital(pd_, lgd=0.45, kind='other retail'):
    """Basel II IRB capital requirement K per unit of EAD (BCBS, 2006, paragraphs 328-330), retail exposures:
    K = LGD [N((G(PD) + sqrt(R) G(0.999)) / sqrt(1 - R)) - PD]; RWA = 12.5 K EAD."""
    pd_ = np.asarray(pd_, float)
    if kind == 'mortgage':
        R = 0.15 + 0 * pd_
    elif kind == 'revolving':
        R = 0.04 + 0 * pd_
    else:
        w = (1 - np.exp(-35 * pd_)) / (1 - np.exp(-35))
        R = 0.03 * w + 0.16 * (1 - w)
    K = lgd * (stats.norm.cdf((stats.norm.ppf(pd_) + np.sqrt(R) * stats.norm.ppf(0.999)) / np.sqrt(1 - R)) - pd_)
    return K, R


def fig_irb(save=True):
    """Expected loss PD x LGD and IRB capital K as functions of PD, LGD = 45%."""
    pds = np.linspace(0.0005, 0.25, 400)
    fig, ax = plt.subplots(figsize=(9.5, 4.0))
    ax.plot(100 * pds, 100 * 0.45 * pds, color=st.Amber, lw=2, label='expected loss EL = PD x LGD')
    for kind, c, ls in [('other retail', st.IDAred, '-'), ('mortgage', st.MainBlue, '--'), ('revolving', st.Forest, '-.')]:
        K, _ = irb_capital(pds, 0.45, kind)
        ax.plot(100 * pds, 100 * K, color=c, lw=2.2, ls=ls, label=f'capital K, {kind}')
    ax.set_xlabel('PD (%)')
    ax.set_ylabel('% of EAD')
    st.legend_outside_bottom(ax, ncol=2, y=-0.2)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch12_irb')


def conditional_pd(pd_, y, rho=0.2):
    """One-factor Gaussian model (Vasicek): borrower i defaults if sqrt(rho) Y + sqrt(1 - rho) e_i < N^-1(PD); given the
    state of the economy Y = y, PD(y) = N((N^-1(PD) - sqrt(rho) y) / sqrt(1 - rho)) (as in the SFE Quantlet SFEdefaproba)."""
    return stats.norm.cdf((stats.norm.ppf(pd_) - np.sqrt(rho) * y) / np.sqrt(1 - rho))


def fig_conditional_pd(rho=0.2, save=True):
    """Conditional PD against the unconditional PD for a bad (y = -3), a typical (y = 0) and a good (y = 3) state of
    the economy, rho = 0.2 (SFEdefaproba)."""
    x = np.linspace(0.001, 0.999, 400)
    fig, ax = plt.subplots(figsize=(9.5, 4.0))
    for y, c, ls, lab in [(-3, st.IDAred, '-', 'bad state of the economy, y = -3'), (0, st.MainBlue, '--', 'typical state, y = 0'),
                          (3, st.Forest, '-.', 'good state, y = 3')]:
        ax.plot(x, conditional_pd(x, y, rho), color=c, lw=2.2, ls=ls, label=lab)
    ax.set_xlabel('unconditional PD')
    ax.set_ylabel('PD given the state y')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    st.legend_outside_bottom(ax, ncol=3, y=-0.2)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch12_conditional_pd')
    return {f'{p}': {f'{y}': float(conditional_pd(p, y, rho)) for y in (-3, 0, 3)} for p in (0.01, 0.02, 0.05)}

## Run

In [ ]:
S = sgc_scorecard()
V = validation(S)
print({k: v for k, v in V.items() if not isinstance(v, dict)})
print(V['cm50'], V['cm_cost'])
fig_roc(S)
fig_cap(S)
print(fig_ks(S))
print(fig_cost_cutoff(S))
res = cv_auc()
print(fig_cv_auc(res))
T = taiwan_model()
cal = fig_calibration(T)
print({k: v for k, v in cal.items() if k != 'groups'})
fig_irb()
print(fig_conditional_pd())

![sfm_ch12_roc](./sfm_ch12_roc.png)

Output: `sfm_ch12_roc.pdf`

![sfm_ch12_cap](./sfm_ch12_cap.png)

Output: `sfm_ch12_cap.pdf`

![sfm_ch12_ks](./sfm_ch12_ks.png)

Output: `sfm_ch12_ks.pdf`

![sfm_ch12_cost_cutoff](./sfm_ch12_cost_cutoff.png)

Output: `sfm_ch12_cost_cutoff.pdf`

![sfm_ch12_cv_auc](./sfm_ch12_cv_auc.png)

Output: `sfm_ch12_cv_auc.pdf`

![sfm_ch12_calibration](./sfm_ch12_calibration.png)

Output: `sfm_ch12_calibration.pdf`

![sfm_ch12_irb](./sfm_ch12_irb.png)

Output: `sfm_ch12_irb.pdf`

![sfm_ch12_conditional_pd](./sfm_ch12_conditional_pd.png)

Output: `sfm_ch12_conditional_pd.pdf`